<a href="https://colab.research.google.com/github/anggaa0519/data-science-2026/blob/main/Pertemuan6_Angga_Anggieanie_250401020172.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tugas Pertemuan 6 — Persiapan Data
## Pipeline Preprocessing End-to-End: Dataset Titanic

| | |
|---|---|
| **Nama** | Angga Anggieanie |
| **NIM** | 250401020172 |
| **Kelas** | IF401 |
| **Mata Kuliah** | Pengantar Data Science (200302305) |
| **Pertemuan** | 6 — Persiapan Data |


## Langkah 1 — Load & EDA Singkat

Memuat dataset Titanic, memilih kolom yang dipakai, lalu memeriksa missing values, tipe data, dan distribusi kelas target `survived`.

In [7]:
import pandas as pd, numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = sns.load_dataset('titanic')

# Pilih kolom yang akan digunakan
cols = ['pclass','sex','age','sibsp','parch','fare','embarked','survived']
df = df[cols].copy()

print('Shape:', df.shape)
print()
print(df.dtypes)
print()
print('Missing values:')
print(df.isnull().sum())
print()
print('Distribusi target:')
print(df['survived'].value_counts(normalize=True).round(3))

Shape: (891, 8)

pclass        int64
sex          object
age         float64
sibsp         int64
parch         int64
fare        float64
embarked     object
survived      int64
dtype: object

Missing values:
pclass        0
sex           0
age         177
sibsp         0
parch         0
fare          0
embarked      2
survived      0
dtype: int64

Distribusi target:
survived
0    0.616
1    0.384
Name: proportion, dtype: float64


In [8]:
df.head()

,pclass,sex,age,sibsp,parch,fare,embarked,survived
0,3,male,22.0,1,0,7.2500,S,0
1,1,female,38.0,1,0,71.2833,C,1
2,3,female,26.0,0,0,7.9250,S,1
3,1,female,35.0,1,0,53.1000,S,1
4,3,male,35.0,0,0,8.0500,S,0


**Interpretasi:** Dataset berisi 891 baris dan 8 kolom. Ada dua masalah missing values: `age` (177 NaN, ~20% data) dan `embarked` (2 NaN). Dua kolom bertipe string (`sex`, `embarked`) perlu di-encode. Target `survived` **tidak seimbang** — 61.6% tidak selamat vs 38.4% selamat — sehingga split nanti wajib memakai `stratify=y` agar proporsi kelas terjaga di train dan test.

## Langkah 2 — Handling Missing Values

Mengisi nilai hilang sebelum encoding: **median** untuk `age` (robust terhadap outlier, lebih aman daripada mean karena distribusi usia miring) dan **modus** untuk `embarked` (nilai kategorik paling sering).

In [9]:
print(f"Median age (pengisi): {df['age'].median()}")
print(f"Modus embarked (pengisi): {df['embarked'].mode()[0]}")

# Age: isi dengan median (robust terhadap outlier)
df['age'] = df['age'].fillna(df['age'].median())

# Embarked: isi dengan modus (nilai paling sering)
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

print()
print('Missing setelah handling:')
print(df.isnull().sum())

Median age (pengisi): 28.0
Modus embarked (pengisi): S

Missing setelah handling:
pclass      0
sex         0
age         0
sibsp       0
parch       0
fare        0
embarked    0
survived    0
dtype: int64


**Interpretasi:** 177 nilai `age` diisi median 28.0 tahun dan 2 nilai `embarked` diisi modus 'S' (Southampton, pelabuhan terbanyak). Seluruh kolom kini bebas missing value (semua 0), sehingga aman untuk tahap encoding dan pemodelan — sebagian besar algoritma ML gagal bila masih ada NaN.

## Langkah 3 — Encoding Kategorikal (One-Hot Encoding)

`sex` dan `embarked` adalah data **nominal** (tanpa urutan alami) sehingga di-encode dengan One-Hot Encoding, bukan Label Encoding. `drop_first=True` dipakai untuk menghindari **dummy variable trap** (multikolinearitas sempurna).

In [10]:
df = pd.get_dummies(df,
                    columns=['sex', 'embarked'],
                    drop_first=True,   # hindari dummy variable trap
                    dtype=int)

print('Kolom setelah encoding:')
print(df.columns.tolist())
df.head()

Kolom setelah encoding:
['pclass', 'age', 'sibsp', 'parch', 'fare', 'survived', 'sex_male', 'embarked_Q', 'embarked_S']


,pclass,age,sibsp,parch,fare,survived,sex_male,embarked_Q,embarked_S
0,3,22.0,1,0,7.2500,0,1,0,1
1,1,38.0,1,0,71.2833,1,0,0,0
2,3,26.0,0,0,7.9250,1,0,0,1
3,1,35.0,1,0,53.1000,1,0,0,1
4,3,35.0,0,0,8.0500,0,1,0,1


**Interpretasi:** `sex` menjadi satu kolom biner `sex_male` (1 = laki-laki; perempuan tersirat saat bernilai 0) dan `embarked` menjadi `embarked_Q` dan `embarked_S` (kategori 'C' menjadi baseline yang dihapus `drop_first`). Seluruh fitur kini numerik — siap diproses operasi matematika model ML — tanpa memaksakan urutan palsu pada data nominal.

## Langkah 4 — Train-Test Split (Stratified)

Dataset dibagi 80:20 (proporsi standar untuk ~891 baris) **sebelum scaling** agar test set benar-benar tak terlihat model. `stratify=y` menjaga proporsi kelas `survived`; `random_state=42` membuat hasil reproducible.

In [11]:
from sklearn.model_selection import train_test_split

X = df.drop('survived', axis=1)
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y      # proporsi kelas terjaga
)

print(f'Train: {X_train.shape[0]} baris')
print(f'Test : {X_test.shape[0]} baris')
print()
print('Proporsi survived di Train:')
print(y_train.value_counts(normalize=True).round(3))
print()
print('Proporsi survived di Test:')
print(y_test.value_counts(normalize=True).round(3))

Train: 712 baris
Test : 179 baris

Proporsi survived di Train:
survived
0    0.617
1    0.383
Name: proportion, dtype: float64

Proporsi survived di Test:
survived
0    0.615
1    0.385
Name: proportion, dtype: float64


**Interpretasi:** Data terbagi menjadi 712 baris train dan 179 baris test. Berkat stratifikasi, proporsi kelas di kedua set hampir identik (≈0.616 vs ≈0.384) — tanpa `stratify`, split acak bisa menghasilkan proporsi yang melenceng dan membuat evaluasi tidak konsisten. Split dilakukan **sebelum** scaling, sesuai urutan pipeline yang benar untuk mencegah data leakage.

## Langkah 5 — Feature Scaling (StandardScaler)

StandardScaler (z-score: mean=0, std=1) diterapkan **hanya pada kolom numerik**. Kunci anti-leakage: `fit_transform()` pada training set saja, lalu `transform()` (tanpa fit!) pada test set — test set di-scale memakai μ dan σ milik training set.

In [12]:
from sklearn.preprocessing import StandardScaler

# Hanya kolom numerik; kolom biner hasil OHE TIDAK di-scale
num_cols = ['pclass', 'age', 'sibsp', 'parch', 'fare']

scaler = StandardScaler()

# fit_transform pada training set (belajar mu dan sigma dari sini)
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])

# transform saja pada test set (pakai mu dan sigma dari training!)
X_test[num_cols] = scaler.transform(X_test[num_cols])

print('Mean scaler (dari train):', scaler.mean_.round(2))
print('Std  scaler (dari train):', scaler.scale_.round(2))
print()
print('Contoh X_train setelah scaling:')
print(X_train.head(3).round(3))
print()
print('Verifikasi: mean ~0 dan std ~1 pada kolom numerik train:')
print(X_train[num_cols].agg(['mean','std']).round(3))
print()
print('Data siap dilatih model Machine Learning!')
print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_test : {X_test.shape},  y_test : {y_test.shape}')

Mean scaler (dari train): [ 2.31 29.46  0.49  0.39 31.82]
Std  scaler (dari train): [ 0.83 13.03  1.06  0.84 48.03]

Contoh X_train setelah scaling:
     pclass    age  sibsp  parch   fare  sex_male  embarked_Q  embarked_S
692   0.830 -0.112 -0.465 -0.466  0.514         1           0           1
481  -0.371 -0.112 -0.465 -0.466 -0.663         1           0           1
527  -1.571 -0.112 -0.465 -0.466  3.955         1           0           1

Verifikasi: mean ~0 dan std ~1 pada kolom numerik train:
      pclass    age  sibsp  parch   fare
mean  -0.000  0.000 -0.000 -0.000 -0.000
std    1.001  1.001  1.001  1.001  1.001

Data siap dilatih model Machine Learning!
X_train: (712, 8), y_train: (712,)
X_test : (179, 8),  y_test : (179,)


**Interpretasi:** Lima kolom numerik kini bermean ≈ 0 dan std ≈ 1 pada training set, sehingga fitur berskala besar seperti `fare` (semula 0–512) tidak lagi mendominasi fitur kecil seperti `pclass` (1–3) pada algoritma berbasis jarak (KNN, SVM) maupun gradient descent. Kolom biner hasil OHE dibiarkan 0/1 karena sudah seragam. Statistik test set sengaja tidak persis 0/1 — itu bukti μ dan σ diambil dari train, bukan dari test (tidak ada leakage).

## Kesimpulan

Pipeline preprocessing Titanic selesai dengan urutan yang benar: (1) EDA mengungkap 179 missing value dan kelas target tak seimbang 61.6:38.4; (2) missing diisi median 28.0 (`age`) dan modus 'S' (`embarked`); (3) fitur nominal di-One-Hot-Encode dengan `drop_first=True` menghasilkan 8 fitur numerik; (4) split stratified 80:20 (712 train / 179 test) menjaga proporsi kelas identik; (5) StandardScaler di-fit hanya pada training set lalu diterapkan ke test set. Prinsip terpenting yang dijaga sepanjang pipeline adalah **mencegah data leakage**: split dilakukan sebelum scaling, dan test set hanya pernah disentuh `transform()` — tidak pernah `fit()`. Dataset kini sepenuhnya numerik, seragam skalanya, dan siap untuk pelatihan model Machine Learning.